# Group 3 | Seasonal Climatology

**Predicting Drought Before the Rains Fail**

District-level severe meteorological drought, Tanzania (1981–2025).

## 2. Data quality control

**Objective.** Verify district-month uniqueness, complete temporal keys, rainfall plausibility, missing SPI observations and internal consistency of supplied severe-drought labels.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "outputs"
for name in ["figures", "maps", "metrics", "predictions", "tables"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

import json
import pandas as pd
from IPython.display import display
from group3_climatology.data import load_data, audit_data, missing_by_year
from group3_climatology.config import INPUT, SPI_FIELD

data = load_data(INPUT)
report = audit_data(data, strict=True)
display(pd.DataFrame(report.items(), columns=["Check", "Result"]))
(OUT / "metrics" / "data_quality.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

,Check,Result
0,district_month_records,81000
1,districts,150
2,regions,31
3,first_month,1981-01
4,last_month,2025-12
5,calendar_months,540
6,valid_spi3,80700
7,missing_spi3,300
8,severe_spi3_months,4364
9,extreme_spi3_months,1399


577

### Consistency rule

For each month with defined SPI-3, the source label must satisfy:

$$Y_{d,t}=\mathbb{1}\{\mathrm{SPI3}_{d,t}\leq -1.5\}.$$

Records with undefined SPI-3 are excluded from the drought-frequency denominator. Rainfall values are checked for physically impossible negatives; exceptionally high monthly rainfall totals are flagged for inspection rather than removed automatically.

In [2]:
print("SPI-3 defined:", data[SPI_FIELD].notna().sum())
print("SPI-3 undefined:", data[SPI_FIELD].isna().sum())
print("Rainfall greater than 1000 mm/month:", int(data.chirps3_rain_mm.gt(1000).sum()))
print("Rainfall exactly zero:", int(data.chirps3_rain_mm.eq(0).sum()))
print("Missing SPI-3 by calendar month:")
display(data.loc[data[SPI_FIELD].isna()].groupby(data.month.dt.strftime("%Y-%m")).size().rename("missing_districts").to_frame())

SPI-3 defined: 80700
SPI-3 undefined: 300
Rainfall greater than 1000 mm/month: 24
Rainfall exactly zero: 203
Missing SPI-3 by calendar month:


,missing_districts
month,
1981-01,150
1981-02,150


### Source limitations

A valid SPI-3 label verifies internal consistency with the supplied threshold; it does not validate rainfall against an independent station network. No claim about livestock losses or crop production can be derived from an SPI label alone.

In [3]:
qc_table = pd.DataFrame([report])
qc_table.to_csv(OUT / "metrics" / "data_quality_summary.csv", index=False)
assert report["duplicate_district_month"] == 0
assert report["spi_threshold_label_mismatches"] == 0
print("All required data-integrity checks passed.")

All required data-integrity checks passed.
